# Flight Delay Propagation Network — Data Ingestion & Cleaning
### Section 9.2 · Phase 1 · Weeks 1–2
**Team 31 · CSE 6242**

This notebook covers every step in Section 9.2 of the project plan:
1. Merge all monthly BTS CSV files into one DataFrame
2. Column selection and renaming
3. Null handling (including mandatory TAIL_NUM drop)
4. Type conversions
5. Data validation tests
6. Export a clean Parquet file for Phase 2

---
> **Before running:** place all monthly BTS CSV files in the `Main_Dataset` folder: 
[Click here to open folder](https://gtvault-my.sharepoint.com/:f:/g/personal/sasghar3_gatech_edu/IgCokK2QKuY6R4HRSQYwEjeCAbTXZZ7SObmGXbIgk4xl14w?e=xJ46N7)


## Cell 0 — Install / import libraries
All libraries below come with Anaconda except `pyarrow` (for Parquet export).
Run the pip line once if `pyarrow` is missing.

In [1]:
# Uncomment if pyarrow is not already installed in your Anaconda environment
!pip install pyarrow --quiet

import pandas as pd
import numpy as np
import os
import glob
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded ✓')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

Libraries loaded ✓
pandas  2.3.3
numpy   2.3.5


## Cell 1 — Configuration
Set `DATA_DIR` to wherever your downloaded BTS CSV files live.
Everything else flows from this.

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = r'C:\Users\kamra\OneDrive - Georgia Institute of Technology\Courses\Spring 2026\CSE6242 Project\Main Dataset'
OUTPUT_DIR = r'C:\Users\kamra\OneDrive - Georgia Institute of Technology\Courses\Spring 2026\CSE6242 Project\clean_output'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── BTS column renames ────────────────────────────────────────────────────────
BTS_RENAME = {
    'OP_UNIQUE_CARRIER' : 'OP_CARRIER',
    'TAIL_NUMBER'       : 'TAIL_NUM',
    'SEC_DELAY'         : 'SECURITY_DELAY',
}

# ── All 25 columns ────────────────────────────────────────────────────────────
KEEP_COLS = [
    'FL_DATE',
    'DAY_OF_WEEK',
    'TAIL_NUM',
    'OP_CARRIER',
    'ORIGIN',               # IATA code — e.g. ORD  (used for network nodes)
    'ORIGIN_AIRPORT_ID',    # numeric DOT ID — e.g. 12892
    'DEST',                 # IATA code — e.g. ATL  (used for network nodes)
    'DEST_AIRPORT_ID',      # numeric DOT ID — e.g. 10397
    'ORIGIN_STATE_ABR',
    'CRS_DEP_TIME',
    'DEP_TIME',
    'DEP_DELAY',
    'CRS_ARR_TIME',
    'ARR_TIME',
    'ARR_DELAY',
    'ARR_DEL15',
    'DEP_DEL15',
    'LATE_AIRCRAFT_DELAY',
    'CARRIER_DELAY',
    'WEATHER_DELAY',
    'NAS_DELAY',
    'SECURITY_DELAY',
    'CANCELLED',
    'DIVERTED',
    'ACTUAL_ELAPSED_TIME',
]

DELAY_CAUSE_COLS = [
    'LATE_AIRCRAFT_DELAY',
    'CARRIER_DELAY',
    'WEATHER_DELAY',
    'NAS_DELAY',
    'SECURITY_DELAY',
]

DELAY_THRESHOLD = 15  # FAA definition of delayed

print('Configuration set ✓')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}  ← folder created if it did not exist')
print(f'Columns    : {len(KEEP_COLS)}')
print(f'Rename map : {BTS_RENAME}')

Configuration set ✓
DATA_DIR   : C:\Users\kamra\OneDrive - Georgia Institute of Technology\Courses\Spring 2026\CSE6242 Project\Main Dataset
OUTPUT_DIR : C:\Users\kamra\OneDrive - Georgia Institute of Technology\Courses\Spring 2026\CSE6242 Project\clean_output  ← folder created if it did not exist
Columns    : 25
Rename map : {'OP_UNIQUE_CARRIER': 'OP_CARRIER', 'TAIL_NUMBER': 'TAIL_NUM', 'SEC_DELAY': 'SECURITY_DELAY'}


## Cell 2 — Discover and preview files
Lists all CSV files found in `DATA_DIR` and shows basic info about each before merging.
Run this first to confirm all 11 months are present.

In [3]:
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

if not csv_files:
    raise FileNotFoundError(
        f'No CSV files found in {DATA_DIR}.\n'
        f'Check that DATA_DIR points to the folder with your BTS downloads.'
    )

print(f'Found {len(csv_files)} CSV file(s):\n')
file_summary = []
for f in csv_files:
    # Read just the first row to get column names and file size
    sample = pd.read_csv(f, nrows=1, dtype=str, low_memory=False)
    size_mb = os.path.getsize(f) / 1_048_576
    file_summary.append({
        'file': os.path.basename(f),
        'size_mb': round(size_mb, 1),
        'columns': len(sample.columns),
    })
    print(f'  {os.path.basename(f):50s}  {size_mb:6.1f} MB  {len(sample.columns)} cols')

print(f'\nTotal size: {sum(r["size_mb"] for r in file_summary):.1f} MB')

# Check all files have the same column structure
col_counts = set(r['columns'] for r in file_summary)
if len(col_counts) > 1:
    print('\n⚠️  WARNING: Files have different column counts — check for BTS format changes between months.')
else:
    print(f'\n✓ All files have the same column structure ({col_counts.pop()} cols)')

Found 11 CSV file(s):

  April_2025.csv                                        69.9 MB  25 cols
  Aug_2025.csv                                          72.4 MB  25 cols
  Feb_2025.csv                                          60.5 MB  25 cols
  Jan_2025.csv                                          64.1 MB  25 cols
  Jul_2025.csv                                          76.4 MB  25 cols
  June_2025.csv                                         54.8 MB  25 cols
  Mar_2025.csv                                          71.9 MB  25 cols
  May_2025.csv                                          54.2 MB  25 cols
  Nov_2025.csv                                          68.7 MB  25 cols
  Oct_2025.csv                                          73.2 MB  25 cols
  Sep_2025.csv                                          67.1 MB  25 cols

Total size: 733.2 MB

✓ All files have the same column structure (25 cols)


## Cell 3 — Load and merge all monthly files
Reads each CSV, applies column renaming, keeps only the needed columns,
then concatenates into one DataFrame.

Loading all as `dtype=str` first avoids pandas making wrong type assumptions
across different months (e.g. a month with all-null TAXI_OUT would infer float
while another infers int). Type coercions happen properly in Cell 5.

In [4]:
monthly_frames = []
load_report    = []

for fpath in csv_files:
    fname = os.path.basename(fpath)
    
    # ── Read raw ──────────────────────────────────────────────────────────────
    raw = pd.read_csv(fpath, dtype=str, low_memory=False)
    
    # ── Normalise column names: strip whitespace, uppercase ───────────────────
    raw.columns = [c.strip().upper() for c in raw.columns]
    
    # ── Drop trailing BTS sentinel column (unnamed last col in some exports) ──
    raw = raw.loc[:, ~raw.columns.str.startswith('UNNAMED')]
    
    # ── Apply rename map ──────────────────────────────────────────────────────
    raw = raw.rename(columns=BTS_RENAME)
    
    # ── Check which of our 25 columns are present ─────────────────────────────
    missing = [c for c in KEEP_COLS if c not in raw.columns]
    present = [c for c in KEEP_COLS if c in raw.columns]
    
    if missing:
        print(f'  ⚠️  {fname}: {len(missing)} column(s) not found: {missing}')
        print(f'       Available columns in file: {list(raw.columns)}')
    
    # ── Keep only our 25 columns ──────────────────────────────────────────────
    df = raw[present].copy()
    
    # ── Tag with source file for debugging ────────────────────────────────────
    df['_SOURCE_FILE'] = fname
    
    # ── APPEND TO LIST ─────────────────────────────────
    monthly_frames.append(df)
    
    load_report.append({
        'file':          fname,
        'rows_raw':      len(df),
        'cols_found':    len(present),
        'cols_missing':  len(missing),
        'missing_names': missing,
    })
    
    print(f'  ✓ {fname:50s}  {len(df):>8,} rows  '
          f'{len(present)}/25 cols found'
          + (f'  ⚠️ missing: {missing}' if missing else ''))

# ── Concatenate all months into one DataFrame ─────────────────────────────────
df_raw = pd.concat(monthly_frames, ignore_index=True)

print(f'\n{"─"*65}')
print(f'MERGED DATASET')
print(f'  Total rows    : {len(df_raw):,}')
print(f'  Total columns : {len(df_raw.columns)}')
print(f'  Memory usage  : {df_raw.memory_usage(deep=True).sum() / 1_048_576:.1f} MB (all string dtype)')
print(f'\nColumns in merged DataFrame:')
for col in [c for c in df_raw.columns if c != '_SOURCE_FILE']:
    print(f'  {col}')

  ✓ April_2025.csv                                       583,950 rows  25/25 cols found
  ✓ Aug_2025.csv                                         602,378 rows  25/25 cols found
  ✓ Feb_2025.csv                                         504,884 rows  25/25 cols found
  ✓ Jan_2025.csv                                         539,747 rows  25/25 cols found
  ✓ Jul_2025.csv                                         631,428 rows  25/25 cols found
  ✓ June_2025.csv                                        611,575 rows  25/25 cols found
  ✓ Mar_2025.csv                                         600,872 rows  25/25 cols found
  ✓ May_2025.csv                                         605,648 rows  25/25 cols found
  ✓ Nov_2025.csv                                         570,550 rows  25/25 cols found
  ✓ Oct_2025.csv                                         605,844 rows  25/25 cols found
  ✓ Sep_2025.csv                                         562,439 rows  25/25 cols found

───────────────────────────────

In [5]:
# ── Free monthly frames from RAM immediately after concat ─────────────────────
# monthly_frames holds 11 separate DataFrames still in memory alongside df_raw.
# Deleting it frees ~half the RAM before Cell 4 runs.
del monthly_frames
import gc
gc.collect()

print(f'Monthly frames cleared from memory ✓')

Monthly frames cleared from memory ✓


In [6]:
import psutil
ram = psutil.virtual_memory()
print(f'Total RAM      : {ram.total / 1_073_741_824:.1f} GB')
print(f'Used RAM       : {ram.used / 1_073_741_824:.1f} GB')
print(f'Available RAM  : {ram.available / 1_073_741_824:.1f} GB')
print(f'df_raw size    : {df_raw.memory_usage(deep=True).sum() / 1_073_741_824:.2f} GB')

Total RAM      : 15.9 GB
Used RAM       : 12.8 GB
Available RAM  : 3.1 GB
df_raw size    : 7.84 GB


## Cell 4 — Section 9.2 Step 3 · Null handling
Rules from the planning doc, applied in order:
1. **Drop rows where TAIL_NUM is null or blank** — these cannot be linked into rotation chains
2. **Fill delay cause columns with 0** — BTS leaves these null when no delay occurred (not missing data)
3. **Separate cancelled/diverted flights** — retain for cancellation analysis but exclude from propagation

In [7]:
n_start = len(df_raw)
print(f'Starting rows: {n_start:,}\n')

# ── 3a. Drop null / blank / placeholder TAIL_NUM ─────────────────────────────
# BTS uses blank strings, 'UNKNOW', '000000', '0' as missing indicators
INVALID_TAIL = {'', 'UNKNOW', 'UNKNOWN', '000000', '0', 'N/A', 'NA', 'NONE'}

tail_null_mask = (
    df_raw['TAIL_NUM'].isna() |
    df_raw['TAIL_NUM'].str.strip().str.upper().isin(INVALID_TAIL)
)
n_null_tail = tail_null_mask.sum()
df_raw = df_raw[~tail_null_mask].copy()
print(f'  Dropped {n_null_tail:,} rows with null/invalid TAIL_NUM  '
      f'({n_null_tail/n_start*100:.2f}% of total)')

# ── 3b. Convert numeric columns from string before filling ────────────────────
# (We need CANCELLED and DIVERTED as numbers to filter on them below)
for col in ['CANCELLED', 'DIVERTED']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0).astype(int)

# ── 3c. Split: cancelled/diverted flights held aside, not dropped ─────────────
# retained for cancellation analysis"
cancelled_mask = df_raw['CANCELLED'] == 1
diverted_mask  = df_raw['DIVERTED']  == 1  if 'DIVERTED' in df_raw.columns else pd.Series(False, index=df_raw.index)

df_cancelled = df_raw[cancelled_mask].copy()
df_diverted  = df_raw[~cancelled_mask & diverted_mask].copy()
df_active    = df_raw[~cancelled_mask & ~diverted_mask].copy()

print(f'  Active (non-cancelled, non-diverted): {len(df_active):,}')
print(f'  Cancelled (held for analysis):        {len(df_cancelled):,}')
print(f'  Diverted  (held for analysis):        {len(df_diverted):,}')

# ── 3d. Fill delay cause nulls with 0 on ACTIVE flights only ─────────────────
# Delay cause columns are null when the flight was on time — that means 0, not missing.
for col in DELAY_CAUSE_COLS:
    if col in df_active.columns:
        n_null = df_active[col].isna().sum()
        df_active[col] = pd.to_numeric(df_active[col], errors='coerce').fillna(0)
        if n_null > 0:
            print(f'  Filled {n_null:,} nulls in {col} with 0')

print(f'\nRows available for propagation model: {len(df_active):,}')

Starting rows: 6,419,315

  Dropped 11,648 rows with null/invalid TAIL_NUM  (0.18% of total)
  Active (non-cancelled, non-diverted): 6,307,561
  Cancelled (held for analysis):        82,227
  Diverted  (held for analysis):        17,879
  Filled 4,926,036 nulls in LATE_AIRCRAFT_DELAY with 0
  Filled 4,926,036 nulls in CARRIER_DELAY with 0
  Filled 4,926,036 nulls in WEATHER_DELAY with 0
  Filled 4,926,036 nulls in NAS_DELAY with 0
  Filled 4,926,036 nulls in SECURITY_DELAY with 0

Rows available for propagation model: 6,307,561


In [8]:
df = df_raw.copy()
print(df.head())

                FL_DATE DAY_OF_WEEK TAIL_NUM OP_CARRIER ORIGIN  \
0  4/7/2025 12:00:00 AM           1   N101NN         AA    JFK   
1  4/7/2025 12:00:00 AM           1   N101NN         AA    LAX   
2  4/7/2025 12:00:00 AM           1   N101NN         AA    LAX   
3  4/7/2025 12:00:00 AM           1   N102NN         AA    JFK   
4  4/7/2025 12:00:00 AM           1   N102NN         AA    SNA   

  ORIGIN_AIRPORT_ID DEST DEST_AIRPORT_ID ORIGIN_STATE_ABR CRS_DEP_TIME  \
0             12478  LAX           12892               NY         1725   
1             12892  BOS           10721               CA         2231   
2             12892  JFK           12478               CA         0800   
3             12478  SNA           14908               NY         0930   
4             14908  JFK           12478               CA         1341   

  DEP_TIME DEP_DELAY CRS_ARR_TIME ARR_TIME ARR_DELAY ARR_DEL15 DEP_DEL15  \
0     1728      3.00         2046     2025    -21.00      0.00      0.00   
1     

## Cell 5 — Data Type conversions
- `FL_DATE` → datetime
- `CRS_DEP_TIME`, `DEP_TIME`, `CRS_ARR_TIME`, `ARR_TIME` → proper time representation
- All numeric delay/elapsed columns → float
- `DAY_OF_WEEK`, `DEP_DEL15`, `ARR_DEL15` → int

In [10]:
df = df_raw.copy()

print('Starting type conversions...\n')

# ══════════════════════════════════════════════════════════════════════════════
# FL_DATE — already parsed to datetime by Cell 4, just verify and extract parts
# ══════════════════════════════════════════════════════════════════════════════
# If still string for any reason, parse it
if df['FL_DATE'].dtype == object:
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], infer_datetime_format=True, errors='coerce')

n_bad_date = df['FL_DATE'].isna().sum()
if n_bad_date > 0:
    print(f'  ⚠️  {n_bad_date:,} rows have unparseable FL_DATE — dropping')
    df = df[df['FL_DATE'].notna()].copy()
print(f'  ✓ FL_DATE: datetime confirmed')
print(f'    Range: {df["FL_DATE"].min().date()}  →  {df["FL_DATE"].max().date()}')

# ══════════════════════════════════════════════════════════════════════════════
# TIME FIELDS — BTS hhmm local time (e.g. 1725 = 5:25pm, 0800 = 8:00am)
# Stored as integers in the data — confirmed from Cell 4 output
# We keep the original hhmm as-is and add a _MIN version for arithmetic
# ══════════════════════════════════════════════════════════════════════════════
def hhmm_to_minutes(series):
    """
    Convert BTS hhmm integer to minutes since midnight.
    Examples: 800 -> 480, 1725 -> 1045, 2400 -> 1440 (BTS midnight convention)
    Returns pd.NA for nulls or out-of-range values.
    """
    s = pd.to_numeric(series, errors='coerce')
    hours  = (s // 100).astype('Int64')
    mins   = (s  % 100).astype('Int64')
    result = hours * 60 + mins
    # BTS uses 2400 for midnight end-of-day — valid, maps to 1440 minutes
    result = result.where((result >= 0) & (result <= 1440), other=pd.NA)
    return result

TIME_HHMM_COLS = ['CRS_DEP_TIME', 'DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME']
for col in TIME_HHMM_COLS:
    if col in df.columns:
        df[col]          = pd.to_numeric(df[col], errors='coerce')  # keep as hhmm int
        df[col + '_MIN'] = hhmm_to_minutes(df[col])                 # add minutes version
        n_null = df[col + '_MIN'].isna().sum()
        print(f'  ✓ {col}: hhmm int kept  |  {col}_MIN added  |  {n_null:,} nulls')

# ══════════════════════════════════════════════════════════════════════════════
# SIGNED DELAY COLUMNS — DEP_DELAY and ARR_DELAY
# Already float from Cell 4 (confirmed: -21.00, -7.00 etc.)
# Negative = early departure/arrival — must preserve the sign
# Fill nulls with 0 only (cancelled flights may have null here)
# ══════════════════════════════════════════════════════════════════════════════
for col in ['DEP_DELAY', 'ARR_DELAY']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        print(f'  ✓ {col}: signed float  '
              f'(min={df[col].min():.0f}, max={df[col].max():.0f})')

# ══════════════════════════════════════════════════════════════════════════════
# DELAY CAUSE COLUMNS — NaN in Cell 4 output means flight was on time = 0
# Never negative by BTS definition ("in Minutes")
# ══════════════════════════════════════════════════════════════════════════════
for col in DELAY_CAUSE_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)

print(f'  ✓ Delay cause columns: NaN filled with 0, clipped to >= 0')
print(f'    {DELAY_CAUSE_COLS}')

# ══════════════════════════════════════════════════════════════════════════════
# ELAPSED TIME — plain positive minutes
# Already float from Cell 4 (confirmed: 357.00, 325.00 etc.)
# ══════════════════════════════════════════════════════════════════════════════
if 'ACTUAL_ELAPSED_TIME' in df.columns:
    df['ACTUAL_ELAPSED_TIME'] = pd.to_numeric(
        df['ACTUAL_ELAPSED_TIME'], errors='coerce'
    ).clip(lower=0)
    print(f'  ✓ ACTUAL_ELAPSED_TIME: float minutes '
          f'(max={df["ACTUAL_ELAPSED_TIME"].max():.0f} min)')

# ══════════════════════════════════════════════════════════════════════════════
# BINARY FLAGS — confirmed as 0.00 floats in Cell 4, convert to clean int 0/1
# DEP_DEL15 / ARR_DEL15: "15 Minutes or More (1=Yes)"
# CANCELLED / DIVERTED:  "Flight Indicator (1=Yes)"
# ══════════════════════════════════════════════════════════════════════════════
BINARY_COLS = ['CANCELLED', 'DIVERTED', 'DEP_DEL15', 'ARR_DEL15']
for col in BINARY_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
print(f'  ✓ Binary flags converted to int 0/1: {BINARY_COLS}')

# ══════════════════════════════════════════════════════════════════════════════
# INTEGER ID AND CATEGORICAL COLUMNS
# ORIGIN_AIRPORT_ID / DEST_AIRPORT_ID: confirmed as integers (12478, 12892 etc.)
# DAY_OF_WEEK: 1=Monday ... 7=Sunday (BTS definition)
# ══════════════════════════════════════════════════════════════════════════════
for col in ['ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID', 'DAY_OF_WEEK']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
print(f'  ✓ ID and categorical columns: Int64')

# ══════════════════════════════════════════════════════════════════════════════
# DERIVED COLUMNS for Phase 2
# ══════════════════════════════════════════════════════════════════════════════
df['FL_DATE_ONLY'] = df['FL_DATE'].dt.date      # for groupby in tail chains
df['MONTH']        = df['FL_DATE'].dt.month
df['YEAR']         = df['FL_DATE'].dt.year
df['IS_WEEKEND']   = df['DAY_OF_WEEK'].isin([6, 7]).astype(int)  # 6=Sat, 7=Sun

print(f'  ✓ Derived columns added: FL_DATE_ONLY, MONTH, YEAR, IS_WEEKEND')

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{"─"*60}')
print(f'TYPE CONVERSION COMPLETE')
print(f'  Shape      : {df.shape}')
print(f'  Date range : {df["FL_DATE"].min().date()} → {df["FL_DATE"].max().date()}')
print(f'\nFinal dtypes:')
print(df.dtypes.to_string())

Starting type conversions...

  ✓ FL_DATE: datetime confirmed
    Range: 2025-01-01  →  2025-11-30
  ✓ CRS_DEP_TIME: hhmm int kept  |  CRS_DEP_TIME_MIN added  |  0 nulls


KeyboardInterrupt: 

## Cell 6 — Data validation
Three checks from the planning doc:
1. ARR_TIME > DEP_TIME for non-overnight flights
2. Flag overnight legs
3. Geographic chain consistency: DEST of flight N == ORIGIN of flight N+1

Plus additional sanity checks that catch common BTS data quality issues.

In [ ]:
validation_log = {}  # accumulate all check results for the summary at end

# ══════════════════════════════════════════════════════════════════════════════
# CHECK 1: ARR_TIME_MIN > DEP_TIME_MIN for non-overnight flights
# An overnight flight departs before midnight and arrives after — DEP_TIME_MIN
# will be > ARR_TIME_MIN (e.g. departs 23:00 = 1380 min, arrives 01:00 = 60 min).
# We define overnight as ARR_TIME_MIN < DEP_TIME_MIN.
# ══════════════════════════════════════════════════════════════════════════════
if 'DEP_TIME_MIN' in df.columns and 'ARR_TIME_MIN' in df.columns:
    has_times = df['DEP_TIME_MIN'].notna() & df['ARR_TIME_MIN'].notna()
    
    overnight_mask = has_times & (df['ARR_TIME_MIN'] < df['DEP_TIME_MIN'])
    df['IS_OVERNIGHT'] = overnight_mask.astype(int)
    
    n_overnight = overnight_mask.sum()
    pct = n_overnight / len(df) * 100
    
    # For non-overnight flights, check that ARR > DEP
    non_overnight = has_times & ~overnight_mask
    bad_time = non_overnight & (df['ARR_TIME_MIN'] <= df['DEP_TIME_MIN'])
    n_bad_time = bad_time.sum()
    
    validation_log['CHECK 1 — Overnight flights flagged'] = (
        f'{n_overnight:,} rows ({pct:.2f}%) flagged IS_OVERNIGHT=1'
    )
    validation_log['CHECK 1 — ARR > DEP violations (non-overnight)'] = (
        f'{n_bad_time:,} rows where ARR_TIME_MIN <= DEP_TIME_MIN on non-overnight flights'
    )
    
    if n_bad_time > 0:
        print(f'⚠️  {n_bad_time:,} non-overnight flights where ARR_TIME <= DEP_TIME')
        print(df[bad_time][['FL_DATE', 'TAIL_NUM', 'ORIGIN', 'DEST',
                             'DEP_TIME', 'ARR_TIME']].head(5).to_string(index=False))
    else:
        print(f'✓ CHECK 1: All non-overnight ARR_TIME > DEP_TIME')
    print(f'  Overnight legs flagged: {n_overnight:,} ({pct:.2f}%)')
else:
    print('  SKIP CHECK 1: DEP_TIME_MIN / ARR_TIME_MIN not available')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 2: Geographic chain consistency
# DEST of flight N must equal ORIGIN of flight N+1 for the same aircraft on the same day.
# Broken chains = data gaps or tail number reuse between operators.
# We FLAG broken links (not drop them) so the team can decide per-chain.
# ══════════════════════════════════════════════════════════════════════════════

# Sort as the tail-chain algorithm will: by aircraft + date + dep time
df = df.sort_values(['TAIL_NUM', 'FL_DATE_ONLY', 'DEP_TIME']).reset_index(drop=True)

# For each flight, get the DEST of the prior flight by the same aircraft on the same day
grp = df.groupby(['TAIL_NUM', 'FL_DATE_ONLY'])
df['PREV_DEST'] = grp['DEST'].shift(1)   # what DEST was the prior leg?

# A broken link: prior leg exists AND prior DEST ≠ this flight's ORIGIN
has_prior    = df['PREV_DEST'].notna()
broken_chain = has_prior & (df['PREV_DEST'] != df['ORIGIN'])
df['CHAIN_BREAK'] = broken_chain.astype(int)

n_broken   = broken_chain.sum()
n_links    = has_prior.sum()
pct_broken = n_broken / n_links * 100 if n_links > 0 else 0

validation_log['CHECK 2 — Chain breaks'] = (
    f'{n_broken:,} broken links out of {n_links:,} leg transitions ({pct_broken:.2f}%)'
)

if n_broken > 0:
    print(f'⚠️  CHECK 2: {n_broken:,} chain breaks ({pct_broken:.2f}% of leg transitions)')
    print('  Sample broken chains (PREV_DEST ≠ ORIGIN):')
    sample_breaks = df[broken_chain][['FL_DATE', 'TAIL_NUM', 'PREV_DEST', 'ORIGIN', 'DEST']].head(8)
    print(sample_breaks.to_string(index=False))
    print('  These rows have CHAIN_BREAK=1 — Phase 2 will exclude them from cascade detection.')
else:
    print('✓ CHECK 2: No chain breaks detected')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 3: ORIGIN and DEST are valid IATA codes (3 uppercase letters)
# ══════════════════════════════════════════════════════════════════════════════
import re
iata_pattern = re.compile(r'^[A-Z]{3}$')

bad_origin = ~df['ORIGIN'].astype(str).str.match(iata_pattern)
bad_dest   = ~df['DEST'].astype(str).str.match(iata_pattern)
n_bad_orig = bad_origin.sum()
n_bad_dest = bad_dest.sum()

validation_log['CHECK 3 — Invalid ORIGIN codes'] = f'{n_bad_orig:,}'
validation_log['CHECK 3 — Invalid DEST codes']   = f'{n_bad_dest:,}'

if n_bad_orig > 0:
    print(f'⚠️  CHECK 3: {n_bad_orig:,} rows with non-IATA ORIGIN values:')
    print(df[bad_origin]['ORIGIN'].value_counts().head(10))
elif n_bad_dest > 0:
    print(f'⚠️  CHECK 3: {n_bad_dest:,} rows with non-IATA DEST values')
else:
    print('✓ CHECK 3: All ORIGIN and DEST values are valid 3-letter IATA codes')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 4: Delay cause columns sum ≤ ARR_DELAY
# The five delay causes should not exceed total arrival delay by more than
# a small tolerance (BTS rounding can create minor discrepancies).
# ══════════════════════════════════════════════════════════════════════════════
cause_sum = df[DELAY_CAUSE_COLS].sum(axis=1)
# Only check rows where ARR_DELAY > 0 (delay must have occurred)
delayed = df['ARR_DELAY'] > 0
tolerance = 5  # minutes — BTS has known minor rounding issues
overcount = delayed & (cause_sum > df['ARR_DELAY'] + tolerance)
n_overcount = overcount.sum()

validation_log['CHECK 4 — Delay causes exceed ARR_DELAY'] = (
    f'{n_overcount:,} rows where sum of delay causes > ARR_DELAY + {tolerance} min'
)

if n_overcount > 0:
    print(f'⚠️  CHECK 4: {n_overcount:,} rows where delay causes sum > ARR_DELAY (tolerance={tolerance} min)')
    print(df[overcount][['FL_DATE','ORIGIN','DEST','ARR_DELAY'] + DELAY_CAUSE_COLS].head(5).to_string(index=False))
else:
    print(f'✓ CHECK 4: Delay cause columns consistent with ARR_DELAY (tolerance={tolerance} min)')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 5: DAY_OF_WEEK matches FL_DATE
# BTS fills DAY_OF_WEEK (1=Mon...7=Sun). We can verify against FL_DATE.
# pandas weekday(): 0=Mon...6=Sun → add 1 to match BTS convention.
# ══════════════════════════════════════════════════════════════════════════════
if 'DAY_OF_WEEK' in df.columns:
    derived_dow = df['FL_DATE'].dt.dayofweek + 1  # 1=Mon, 7=Sun
    dow_mismatch = df['DAY_OF_WEEK'].notna() & (df['DAY_OF_WEEK'].astype(float).astype('Int64') != derived_dow)
    n_mismatch = dow_mismatch.sum()
    
    validation_log['CHECK 5 — DAY_OF_WEEK vs FL_DATE'] = (
        f'{n_mismatch:,} rows where DAY_OF_WEEK disagrees with FL_DATE'
    )
    
    if n_mismatch > 0:
        print(f'⚠️  CHECK 5: {n_mismatch:,} rows where DAY_OF_WEEK disagrees with FL_DATE')
        # Overwrite with derived value — FL_DATE is ground truth
        df.loc[dow_mismatch, 'DAY_OF_WEEK'] = derived_dow[dow_mismatch]
        print('  → Corrected DAY_OF_WEEK from FL_DATE for affected rows')
    else:
        print('✓ CHECK 5: DAY_OF_WEEK consistent with FL_DATE across all rows')
else:
    print('  SKIP CHECK 5: DAY_OF_WEEK column not present')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 6: LATE_AIRCRAFT_DELAY only present when ARR_DELAY > 0
# If LATE_AIRCRAFT_DELAY > 0 but ARR_DELAY <= 0, the aircraft arrived early
# despite being assigned a late-aircraft cause — a data inconsistency.
# ══════════════════════════════════════════════════════════════════════════════
late_but_early = (df['LATE_AIRCRAFT_DELAY'] > 0) & (df['ARR_DELAY'] <= 0)
n_late_early = late_but_early.sum()

validation_log['CHECK 6 — LATE_AIRCRAFT_DELAY with non-positive ARR_DELAY'] = (
    f'{n_late_early:,} rows'
)

if n_late_early > 0:
    print(f'⚠️  CHECK 6: {n_late_early:,} rows where LATE_AIRCRAFT_DELAY > 0 but ARR_DELAY <= 0')
    print('  These rows will be excluded from IS_ROTATIONAL_CASCADE detection in Phase 2.')
else:
    print('✓ CHECK 6: LATE_AIRCRAFT_DELAY only present on genuinely delayed flights')

print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECK 7: Coverage across all expected months
# ══════════════════════════════════════════════════════════════════════════════
month_counts = df.groupby(['YEAR', 'MONTH']).size().reset_index(name='flights')
month_counts['year_month'] = month_counts['YEAR'].astype(str) + '-' + month_counts['MONTH'].astype(str).str.zfill(2)

print('CHECK 7 — Monthly flight counts:')
print(month_counts[['year_month', 'flights']].to_string(index=False))

if len(month_counts) < 2:
    print('⚠️  Only 1 month loaded — confirm all CSV files are in DATA_DIR')
else:
    # Flag any month that is more than 30% below the median (likely incomplete download)
    median_flights = month_counts['flights'].median()
    low_months = month_counts[month_counts['flights'] < median_flights * 0.7]
    if len(low_months) > 0:
        print(f'\n⚠️  These months have unusually low flight counts (< 70% of median {median_flights:,.0f}):')
        print(low_months[['year_month', 'flights']].to_string(index=False))
        print('  → Possible incomplete download. Re-check those files.')
    else:
        print(f'\n✓ CHECK 7: All months have consistent flight counts (median: {median_flights:,.0f})')

validation_log['CHECK 7 — Months loaded'] = f'{len(month_counts)} month(s)'

## Cell 7 — Validation summary
All checks in one table. Pass this to the team before moving to Phase 2.

In [ ]:
print('=' * 70)
print('VALIDATION SUMMARY')
print('=' * 70)
for check, result in validation_log.items():
    flag = '⚠️ ' if any(x in result for x in ['broken', 'violat', 'mismatch', 'exceed', 'bad']) else '✓  '
    print(f'{flag} {check}')
    print(f'     {result}')
    print()

print('=' * 70)
print(f'FINAL CLEAN DATASET')
print(f'  Active flights (for propagation model) : {len(df):,}')
print(f'  Cancelled flights (held separately)    : {len(df_cancelled):,}')
print(f'  Diverted flights  (held separately)    : {len(df_diverted):,}')
print(f'  Chain breaks flagged                   : {df["CHAIN_BREAK"].sum():,}')
print(f'  Overnight legs flagged                 : {df["IS_OVERNIGHT"].sum() if "IS_OVERNIGHT" in df.columns else "N/A":}')
print(f'  Date range                             : {df["FL_DATE"].min().date()} → {df["FL_DATE"].max().date()}')
print(f'  Unique aircraft (TAIL_NUM)             : {df["TAIL_NUM"].nunique():,}')
print(f'  Unique carriers (OP_CARRIER)           : {df["OP_CARRIER"].nunique():,}')
print(f'  Columns in clean dataset               : {len(df.columns)}')
print('=' * 70)

## Cell 8 — Export clean data
Saves three files to `OUTPUT_DIR`:
- `flights_clean.parquet` — main file for Phase 2 (fast, typed, compressed)
- `flights_cancelled.parquet` — cancelled flights retained per Section 9.2 Step 3
- `ingestion_report.csv` — per-file load report for documentation

Parquet is strongly preferred over CSV for the merged dataset — it preserves dtypes,
loads 5–10x faster, and is ~70% smaller. pandas reads it with `pd.read_parquet()`.

In [ ]:
# ── Drop internal helper columns before export ────────────────────────────────
drop_internal = ['_SOURCE_FILE', 'PREV_DEST']  # only needed during validation
df_export = df.drop(columns=[c for c in drop_internal if c in df.columns], errors='ignore')

# ── Export main clean file ────────────────────────────────────────────────────
out_main = os.path.join(OUTPUT_DIR, 'flights_clean.parquet')
df_export.to_parquet(out_main, index=False, engine='pyarrow', compression='snappy')
size_mb = os.path.getsize(out_main) / 1_048_576
print(f'✓ Saved flights_clean.parquet  {len(df_export):,} rows  {size_mb:.1f} MB')

# ── Export cancelled flights ──────────────────────────────────────────────────
if len(df_cancelled) > 0:
    out_cancelled = os.path.join(OUTPUT_DIR, 'flights_cancelled.parquet')
    df_cancelled.to_parquet(out_cancelled, index=False, engine='pyarrow', compression='snappy')
    print(f'✓ Saved flights_cancelled.parquet  {len(df_cancelled):,} rows')

# ── Export diverted flights ───────────────────────────────────────────────────
if len(df_diverted) > 0:
    out_diverted = os.path.join(OUTPUT_DIR, 'flights_diverted.parquet')
    df_diverted.to_parquet(out_diverted, index=False, engine='pyarrow', compression='snappy')
    print(f'✓ Saved flights_diverted.parquet   {len(df_diverted):,} rows')

# ── Export load report ────────────────────────────────────────────────────────
report_df = pd.DataFrame(load_report)
out_report = os.path.join(OUTPUT_DIR, 'ingestion_report.csv')
report_df.to_csv(out_report, index=False)
print(f'✓ Saved ingestion_report.csv')

print(f'\nAll outputs in: {os.path.abspath(OUTPUT_DIR)}')
print('\nTo load in Phase 2 notebook:')
print("  df = pd.read_parquet('./data/clean/flights_clean.parquet')")

## Cell 9 — Quick EDA (sanity check before handing to Phase 2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('BTS Data — Ingestion Sanity Check (All Airports)', fontsize=14, fontweight='bold')

# ── Plot 1: Flights per month ─────────────────────────────────────────────────
ax = axes[0, 0]
monthly = df.groupby('MONTH').size().sort_index()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
bars = ax.bar(monthly.index, monthly.values, color='steelblue', edgecolor='white')
ax.set_title('Flights per Month')
ax.set_xlabel('Month')
ax.set_ylabel('Flights')
ax.set_xticks(monthly.index)
ax.set_xticklabels([month_labels[m-1] for m in monthly.index], rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1_000_000:.1f}M'))
for bar, val in zip(bars, monthly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val/1000:.0f}K', ha='center', va='bottom', fontsize=8)

# ── Plot 2: Top 20 origins by flight count (all airports, no filter) ──────────
ax = axes[0, 1]
top_origins = df['ORIGIN'].value_counts().head(20)
top_origins.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Origin Airports by Flight Count')
ax.set_xlabel('Airport')
ax.set_ylabel('Flights')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.tick_params(axis='x', rotation=45)

# ── Plot 3: Distribution of LATE_AIRCRAFT_DELAY (non-zero only) ──────────────
ax = axes[1, 0]
late_nonzero = df[df['LATE_AIRCRAFT_DELAY'] > 0]['LATE_AIRCRAFT_DELAY'].clip(upper=180)
ax.hist(late_nonzero, bins=40, color='darkorange', edgecolor='white', alpha=0.85)
ax.set_title(f'LATE_AIRCRAFT_DELAY Distribution\n(non-zero only, capped 180 min  |  n={len(late_nonzero):,})')
ax.set_xlabel('Minutes')
ax.set_ylabel('Flights')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Plot 4: Delay rate by carrier (all carriers) ──────────────────────────────
ax = axes[1, 1]
carrier_delay = df.groupby('OP_CARRIER').apply(
    lambda x: (x['DEP_DELAY'] >= DELAY_THRESHOLD).mean() * 100
).sort_values(ascending=False)
carrier_delay.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Departure Delay Rate by Carrier (≥15 min)')
ax.set_xlabel('Carrier')
ax.set_ylabel('Delay Rate (%)')
ax.tick_params(axis='x', rotation=45)
for i, val in enumerate(carrier_delay.values):
    ax.text(i, val + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'ingestion_eda.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'\n✓ EDA chart saved to: {os.path.join(OUTPUT_DIR, "ingestion_eda.png")}')
print(f'\nDataset summary:')
print(f'  Total flights     : {len(df):,}')
print(f'  Unique origins    : {df["ORIGIN"].nunique():,}')
print(f'  Unique carriers   : {df["OP_CARRIER"].nunique():,}')
print(f'  Unique aircraft   : {df["TAIL_NUM"].nunique():,}')
print(f'  Flights with late aircraft delay > 0 : {(df["LATE_AIRCRAFT_DELAY"] > 0).sum():,}')
print(f'  Overall dep delay rate (≥15 min)     : {(df["DEP_DELAY"] >= DELAY_THRESHOLD).mean()*100:.1f}%')

---
## Handoff checklist for Phase 2 - Model Propogation

Before passing `flights_clean.parquet` to the Phase 2 team, confirm:

- [ ] All expected months are present (Check 7 shows no low-count months)
- [ ] Zero chain breaks OR chain breaks reviewed and accepted
- [ ] ORIGIN/DEST are valid IATA codes (Check 3 passed)
- [ ] `TAIL_NUM` has no nulls (enforced in Cell 4)
- [ ] `FL_DATE` is datetime, `DEP_TIME_MIN` and `ARR_TIME_MIN` are numeric minutes
- [ ] `IS_OVERNIGHT` and `CHAIN_BREAK` columns present for Phase 2 filtering
- [ ] Validation summary printed with no unresolved ⚠️ items

Phase 2 entry point:
```python
import pandas as pd
df = pd.read_parquet('./data/clean/flights_clean.parquet')
```

In [14]:
#test load
import pandas as pd
df = pd.read_parquet('flights_clean.parquet')

print(len(df))
df.head(10)


6407667


,FL_DATE,DAY_OF_WEEK,TAIL_NUM,OP_CARRIER,ORIGIN,ORIGIN_AIRPORT_ID,DEST,DEST_AIRPORT_ID,ORIGIN_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DEL15,DEP_DEL15,LATE_AIRCRAFT_DELAY,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,CANCELLED,DIVERTED,ACTUAL_ELAPSED_TIME,CRS_DEP_TIME_MIN,DEP_TIME_MIN,CRS_ARR_TIME_MIN,ARR_TIME_MIN,FL_DATE_ONLY,MONTH,YEAR,IS_WEEKEND,IS_OVERNIGHT,CHAIN_BREAK
0,2025-01-01,3,188NV,G4,FLL,11697,PBG,14025,FL,1000,955.00,-5.00,1319,1247.00,-32.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,172.00,600,595,799,767,2025-01-01,1,2025,0,0,0
1,2025-01-01,3,188NV,G4,PBG,14025,FLL,11697,NY,1409,1406.00,-3.00,1752,1735.00,-17.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,209.00,849,846,1072,1055,2025-01-01,1,2025,0,0,0
2,2025-01-02,4,188NV,G4,FLL,11697,LEX,12945,FL,600,557.00,-3.00,828,807.00,-21.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,130.00,360,357,508,487,2025-01-02,1,2025,0,0,0
3,2025-01-02,4,188NV,G4,LEX,12945,FLL,11697,KY,918,909.00,-9.00,1145,1124.00,-21.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,135.00,558,549,705,684,2025-01-02,1,2025,0,0,0
4,2025-01-02,4,188NV,G4,FLL,11697,USA,12544,FL,1235,1231.00,-4.00,1435,1436.00,1.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,125.00,755,751,875,876,2025-01-02,1,2025,0,0,0
5,2025-01-02,4,188NV,G4,USA,12544,FLL,11697,NC,1525,1542.00,17.00,1726,1741.00,15.00,1,1,0.00,15.00,0.00,0.00,0.00,0,0,119.00,925,942,1046,1061,2025-01-02,1,2025,0,0,0
6,2025-01-02,4,188NV,G4,FLL,11697,TYS,15412,FL,1816,1844.00,28.00,2020,2051.00,31.00,1,1,15.00,13.00,0.00,3.00,0.00,0,0,127.00,1096,1124,1220,1251,2025-01-02,1,2025,0,0,0
7,2025-01-02,4,188NV,G4,TYS,15412,FLL,11697,TN,2110,2142.00,32.00,2320,5.00,45.00,1,1,31.00,1.00,0.00,13.00,0.00,0,0,143.00,1270,1302,1400,5,2025-01-02,1,2025,0,1,0
8,2025-01-03,5,188NV,G4,FLL,11697,ABE,10135,FL,720,720.00,0.00,1014,1003.00,-11.00,0,0,0.00,0.00,0.00,0.00,0.00,0,0,163.00,440,440,614,603,2025-01-03,1,2025,0,0,0
9,2025-01-03,5,188NV,G4,ABE,10135,FLL,11697,PA,1104,1054.00,-10.00,1406,1435.00,29.00,1,0,0.00,0.00,0.00,29.00,0.00,0,0,221.00,664,654,846,875,2025-01-03,1,2025,0,0,0
